# Create Modelfile
This section creates a reusable model definition so citizen developers and DevOps engineers get consistent, production-minded responses.
Key definitions:
- `Modelfile`: Instruction file that tells Ollama how to build and run a custom model.
- `FROM`: Base model to inherit from.
- `PARAMETER`: Runtime tuning controls for generation behavior.
- `SYSTEM`: High-priority behavioral instructions for model responses.

In [ ]:
import os

modelfile_text = '''

FROM llama3.2-vision:latest
PARAMETER temperature 0.7
PARAMETER top_k 90
PARAMETER top_p 1
PARAMETER num_ctx 10000
PARAMETER temperature 0.1
PARAMETER repeat_last_n 70
PARAMETER seed 0

SYSTEM """
You are NetOps/Telecom Principal Engineer Copilot.

Scope & Expertise:
- Deep expertise in IP networking and Telecommunications: routing/switching, MPLS, BGP/OSPF/IS-IS, EVPN/VXLAN, QoS, multicast, transport/optical concepts, SIP/IMS, RTP/SRTP, VoIP, SBCs, call flows, dial plans, numbering, and troubleshooting.
- Strong software engineering skills: Python, Java (Spring), JavaScript/TypeScript, APIs, automation, testing, and CI/CD.

Behavior:
- Be accurate, pragmatic, and production-minded. Prefer actionable guidance over theory.
- Keep responses thorough but not verbose: default to concise explanations, then provide bullets/steps/examples only as needed.
- If you must assume something, say so briefly and choose the safest reasonable assumption.
- Ask at most 1–3 clarifying questions only when required to avoid a wrong answer; otherwise proceed with best-effort and note what to verify.

How to answer:
1) Start with the direct answer or recommended approach.
2) Provide the key reasoning in short bullets.
3) Provide concrete commands, config snippets, code, or checklists when useful.
4) Include verification steps and common pitfalls.
5) When presenting code/config, keep it minimal, correct, and runnable. Prefer idiomatic style and clear naming.

Vision inputs (images/screenshots/diagrams):
- If an image is provided, describe what you observe, extract relevant details (labels, alarms, counters, logs), and map them to likely root causes.
- When uncertain, list the top plausible interpretations and what evidence would confirm each.
"""
'''

with open ('Modelfile', 'w', encoding='utf-8') as f:
    f.write(modelfile_text)
print("Modelfile created at.", os.path.abspath('Modelfile'))

import subprocess, shlex
def run_command(command):
    print('$', command)
    process = subprocess.run(shlex.split(command), check=True, text=True, capture_output=True)
    print(process.stdout)
    print(process.stderr)
    return process

## Build Local Model
This step builds the custom model from the Modelfile and verifies it is available for local use.
Key definitions:
- `ollama create`: Builds a model artifact from a Modelfile.
- `MODEL_NAME`: Reusable local model tag used in later API calls.
- `subprocess.run`: Python API that executes shell commands from code.
- `capture_output=True`: Captures stdout and stderr for debugging and validation.

In [ ]:
import subprocess, shlex, os, textwrap

MODEL_NAME = "netops-llama3.2-vision"
MODEL_FILE = "Modelfile"

def run(cmd):
    print('$', cmd)
    process = subprocess.run(shlex.split(cmd), check=True, text=True, capture_output=True)
    if process.stdout.strip():
        print(process.stdout)
    if process.stderr.strip():
        print(process.stderr)
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd, output=process.stdout, stderr=process.stderr)
    return process

print("Working directory:", os.getcwd())
run("ls -la")
run(f"ollama create {MODEL_NAME} -f {MODEL_FILE}")
run(f"ollama list")

# Check Environment
This quick check confirms your Python runtime and platform so setup issues are easier to troubleshoot.
Key definitions:
- `Runtime`: Active Python interpreter executing the notebook.
- `Platform`: OS and architecture details that can affect behavior.
- `Dependency drift`: Mismatch between expected and actual environment versions.

In [ ]:
import sys, platform, json, textwrap, os, time
print("Python:", sys.version)
print("Platform:", platform.platform())

# Ensure that Ollama is running
This verifies the Ollama service is reachable before sending model requests.
Key definitions:
- `localhost:11434`: Default local Ollama API endpoint.
- `Health check`: Small request used to confirm service availability.
- `URLError`: Network-level failure (service down, unreachable host, etc.).
- `timeout`: Maximum wait time before treating a request as failed.

In [ ]:
import urllib.request, urllib.error

def ollama_up(host="http://localhost:11434"):
    try:
        with urllib.request.urlopen(host, timeout=5) as response:
            return True, response.status
    except urllib.error.URLError as e:
        return False, str(e)

ok, info = ollama_up()
print("Ollama is up and running.", ok, "|", info)


# List the local models
This lists locally available models so you can confirm the expected model name before inference.
Key definitions:
- `Tag`: User-facing model identifier such as `name:latest`.
- `Digest`: Content hash that uniquely identifies model bytes.
- `Quantization`: Compression level trading quality for memory and speed.
- `Parameter size`: Approximate model scale (for example, `8B`, `20B`).

In [ ]:
import urllib.request, json

HOST = "http://localhost:11434"

def normalize_host(host: str) -> str:
    return host.rstrip("/")

def ollama_tags(host=HOST):
    host = normalize_host(host)
    request_url = urllib.request.Request(f"{host}/api/tags", method="GET")
    with urllib.request.urlopen(request_url) as response:
        return json.loads(response.read().decode("utf-8"))

tags = ollama_tags()
print("Models in Ollama:", len(tags.get("models", [])))
print("Ollama tags:", tags)


## Send Text Chat Request
This section demonstrates a structured text chat call to validate model behavior and API connectivity.
Key definitions:
- `messages`: Ordered conversation list sent to `/api/chat`.
- `role`: Speaker type (`system`, `user`, or `assistant`).
- `payload`: JSON body posted to the model API.
- `stream=False`: Returns one complete response instead of token streaming.
- `options`: Optional generation controls (for example `num_predict`, `temperature`).

In [ ]:
import urllib.request, urllib.error, json

HOST = "http://localhost:11434"
MODEL_NAME = "netops-llama3.2-vision"

def normalize_host(host: str) -> str:
    return host.rstrip("/")

def chat(messages, model=MODEL_NAME, host=HOST, stream=False, options=None):
    """Send messages to Ollama chat API. Messages can be a string or list of message dicts."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    payload = {
        "model": model,
        "messages": messages,
        "stream": stream,
    }
    if options:
        payload["options"] = options

    host = normalize_host(host)
    request_data = json.dumps(payload).encode("utf-8")
    request_url = urllib.request.Request(
        f"{host}/api/chat",
        data=request_data,
        method="POST",
        headers={"Content-Type": "application/json"},
    )

    try:
        with urllib.request.urlopen(request_url) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="ignore") if hasattr(e, "read") else ""
        raise RuntimeError(f"Ollama chat failed: HTTP {e.code} {e.reason}. Body: {body}") from e

# Example: Multi-turn conversation with BGP configuration
response = chat(
    [
        {"role": "system", "content": "You are a network engineer. Always use Loopback0 for BGP router-id and enable log-neighbor-changes."},
        {"role": "user", "content": "Configure BGP for AS 65001 with neighbor 192.168.1.2 in AS 65002"},
        {"role": "assistant", "content": "interface Loopback0\n description BGP_ROUTER_ID\n ip address 10.1.1.1 255.255.255.255\n!\nrouter bgp 65001\n bgp router-id 10.1.1.1\n bgp log-neighbor-changes\n neighbor 192.168.1.2 remote-as 65002\n neighbor 192.168.1.2 activate"},
    ]
)

print("Response:", response["message"]["content"])

## Interpret Logs or Pasted Data
This section defines a helper that converts operational text data into concise, actionable JSON insights.
Key definitions:
- `Prompt template`: Reusable structure used to build user input.
- `Context`: Extra situational details that improve model relevance.
- `Structured output`: Predictable format designed for automation workflows.
- `Actionable insight`: Recommendation that can be executed by an operator or pipeline.

In [ ]:
import json, urllib.request, textwrap

HOST = "http://localhost:11434"
MODEL_NAME = "netops-llama3.2-vision"

INTERPRET_INSTRUCTIONS = """
Interpret the pasted data. Output in a structured JSON format with key insights, likely root causes, and recommended next steps.
1) Summary of what the data shows (2-4 bullets)
2) Key details that support the summary (bullets, logs, counters, labels)
3) Likely root causes (bullets, logs, counters, labels)
4) Recommended next steps (bullets, actions, mitigations)
5) Any uncertainties or assumptions (bullets)
Keep the output concise and focused on actionable insights.
"""

def interpret_data(pasted_data: str, context, str = "") -> str:
    user_content = textwrap.dedent(f"""
    Context:
    {context}
    Data:
    {pasted_data}
    Instructions:
    {INTERPRET_INSTRUCTIONS}
    """)

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": user_content}
        ],
        "stream": False,
    }

    request_data = json.dumps(payload).encode("utf-8")
    request_url = urllib.request.Request(
        f"{HOST}/api/chat",
        data=request_data,
        method="POST",
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request_url) as response:
        return json.loads(response.read().decode("utf-8"))["message"]["content"]
        

## Run Sample Log Analysis
This runs the log interpretation helper on sample network events to show expected output format.
Key definitions:
- `BGP adjacency`: Neighbor relationship required for route exchange.
- `Hold time expired`: BGP session dropped because keepalives were missed.
- `Link flap`: Interface repeatedly transitions up and down.
- `Correlation`: Matching multiple events over time to infer root cause.

In [ ]:
sample_data = """
Feb 24 10:21:04 edge1 bgpd[1234]: %BGP-5-ADJCHANGE: neighbor 10.10.10.2 Down BGP Notification sent
Feb 24 10:21:05 edge1 bgpd[1234]: %BGP-3-NOTIFICATION: received from neighbor 10.10.10.2 4/0 (hold time expired) 0 bytes
Feb 24 10:21:08 edge1 kernel: eth0: Link is Down
Feb 24 10:21:12 edge1 kernel: eth0: Link is Up - 10Gbps Full Duplex
"""

print(interpret_data(sample_data, context="Customer reports intermittent route loss to upstream; this is a PE in MPLS core."))


## Analyze Images with the Model
This section adds an image-aware helper for extracting structured observations from screenshots or diagrams.
Key definitions:
- `Multimodal`: Model can process both text and image inputs.
- `Base64`: Binary-to-text encoding used to send image bytes in JSON.
- `option_profile`: Named preset for latency/quality trade-offs.
- `retries`: Automatic reattempts for transient failures.
- `timeout_seconds`: Per-request wait limit before aborting.

In [ ]:
import base64, pathlib, json, urllib.request, urllib.error, socket, time

OPTION_PROFILES = {
    # Lower-latency profile: shorter generation and lower temperature.
    "fast": {"num_predict": 96, "temperature": 0.0, "top_p": 0.9},
    # Balanced default profile for moderate latency and quality.
    "balanced": {"num_predict": 192, "temperature": 0.1, "top_p": 0.95},
    # Higher-quality profile for richer responses at higher latency.
    "quality": {"num_predict": 384, "temperature": 0.2, "top_p": 0.95},
}

def normalize_host(host: str) -> str:
    return host.rstrip("/")

def chat_with_image(
    image_path: str,
    user_text: str,
    host=HOST,
    model=MODEL_NAME,
    timeout_seconds: int = 420,
    retries: int = 2,
    retry_backoff_seconds: int = 5,
    option_profile: str | None = "fast",
    options: dict | None = None,
) -> str:
    path = pathlib.Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    image_bytes = path.read_bytes()
    image_b64 = base64.b64encode(image_bytes).decode("utf-8")

    payload = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": user_text,
                "images": [image_b64],
            }
        ],
        "stream": False,
    }

    selected_options = {}
    if option_profile:
        if option_profile not in OPTION_PROFILES:
            supported = ", ".join(sorted(OPTION_PROFILES.keys()))
            raise ValueError(f"Unknown option_profile '{option_profile}'. Supported: {supported}")
        selected_options.update(OPTION_PROFILES[option_profile])
    if options:
        selected_options.update(options)
    if selected_options:
        payload["options"] = selected_options

    host = normalize_host(host)
    request_data = json.dumps(payload).encode("utf-8")
    request_url = urllib.request.Request(
        f"{host}/api/chat",
        data=request_data,
        method="POST",
        headers={"Content-Type": "application/json"},
    )

    last_error = None
    for attempt in range(retries + 1):
        try:
            with urllib.request.urlopen(request_url, timeout=timeout_seconds) as response:
                body = json.loads(response.read().decode("utf-8"))
                return body["message"]["content"]
        except TimeoutError as e:
            last_error = e
        except socket.timeout as e:
            last_error = e
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", errors="ignore") if hasattr(e, "read") else ""
            raise RuntimeError(f"Ollama chat_with_image failed: HTTP {e.code} {e.reason}. Body: {detail}") from e
        except urllib.error.URLError as e:
            last_error = e

        if attempt < retries:
            time.sleep(retry_backoff_seconds * (attempt + 1))

    raise TimeoutError(
        f"Timed out after {retries + 1} attempts calling {host}/api/chat with timeout={timeout_seconds}s"
    ) from last_error

## Run Image Analysis Example
This executes an example vision prompt and returns structured output for operational reporting workflows.
Key definitions:
- `JSON-only prompt`: Instruction that constrains response shape for machine-readability.
- `BUS-ID`: PCI bus identifier used to locate a GPU device.
- `Inference latency`: Total time from request send to full response.
- `Profile override`: Per-call tuning that supersedes default generation settings.

In [ ]:
print(
    chat_with_image(
        "/Users/AC05735/DevOps/1a_DevProjects_local/Code/demo/SpringSummit26/llamaDemoPic.webp",
        "Can you give me a JSON response of the GPU number, its temperature, power usage along with BUS-ID?",
        option_profile="fast",
        timeout_seconds=420,
        retries=1,
    )
)